In [1]:
import os
import pandas as pd
import numpy as np
from PIL import Image
from sklearn.model_selection import train_test_split

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
import torchvision.models as models

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"📍 Using device: {device}")

📍 Using device: cuda


In [2]:
# 전처리 & Custom Dataset 정의

transform_train = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

transform_test = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

class CarDataset(Dataset):
    def __init__(self, image_paths, labels=None, transform=None):
        self.image_paths = image_paths
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        image = Image.open(img_path).convert("RGB")
        if self.transform:
            image = self.transform(image)

        if self.labels is not None:
            return image, self.labels[idx]
        else:
            return image

In [3]:
# 🔸 모든 train 이미지 경로 수집
train_dir = "/kaggle/input/hectocar/train"
class_names = sorted(os.listdir(train_dir))  # 총 396 클래스
class_to_idx = {cls_name: idx for idx, cls_name in enumerate(class_names)}

image_paths = []
labels = []

for cls_name in class_names:
    cls_folder = os.path.join(train_dir, cls_name)
    for img_file in os.listdir(cls_folder):
        image_paths.append(os.path.join(cls_folder, img_file))
        labels.append(class_to_idx[cls_name])

# 🔸 Train/Val Split
train_paths, val_paths, train_labels, val_labels = train_test_split(
    image_paths, labels, test_size=0.1, stratify=labels, random_state=42
)

# 🔸 Dataloader
train_dataset = CarDataset(train_paths, train_labels, transform=transform_train)
val_dataset = CarDataset(val_paths, val_labels, transform=transform_test)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False, num_workers=2)

In [4]:
def train_one_epoch(model, loader, criterion, optimizer):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        _, preds = torch.max(outputs, 1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    return running_loss / len(loader), correct / total


def evaluate(model, loader, criterion):
    model.eval()
    val_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)

            val_loss += loss.item()
            _, preds = torch.max(outputs, 1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    return val_loss / len(loader), correct / total

In [5]:
from tqdm import tqdm  # ✅ tqdm 추가

# ✅ 모델 정의
model = models.resnet18(weights=None)
model.fc = nn.Linear(model.fc.in_features, 396)
model = model.to(device)

# ✅ 손실 함수 및 옵티마이저
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=3e-4)

# ✅ 에폭 설정
EPOCHS = 30
best_acc = 0

# ✅ 학습 및 검증 루프
for epoch in range(EPOCHS):
    # ✅ train loop
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in tqdm(train_loader, desc=f"🟦 Training Epoch {epoch+1}", leave=False):
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        _, predicted = torch.max(outputs, 1)
        correct += (predicted == labels).sum().item()
        total += labels.size(0)

    train_loss = running_loss / total
    train_acc = correct / total

    # ✅ validation loop
    model.eval()
    val_running_loss = 0.0
    val_correct = 0
    val_total = 0

    with torch.no_grad():
        for images, labels in tqdm(val_loader, desc=f"🟩 Validating Epoch {epoch+1}", leave=False):
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)

            val_running_loss += loss.item() * images.size(0)
            _, predicted = torch.max(outputs, 1)
            val_correct += (predicted == labels).sum().item()
            val_total += labels.size(0)

    val_loss = val_running_loss / val_total
    val_acc = val_correct / val_total

    # ✅ 출력 및 저장
    print(f"\n📘 Epoch {epoch+1}/{EPOCHS}")
    print(f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f}")
    print(f"Val   Loss: {val_loss:.4f}, Val   Acc: {val_acc:.4f}")

    if val_acc > best_acc:
        best_acc = val_acc
        torch.save(model.state_dict(), "best_model_396.pth")
        print("✅ Best model saved!")



📘 Epoch 1/30
Train Loss: 5.0553, Train Acc: 0.0665
Val   Loss: 3.9806, Val   Acc: 0.1463
✅ Best model saved!



📘 Epoch 2/30
Train Loss: 2.7661, Train Acc: 0.3750
Val   Loss: 2.9833, Val   Acc: 0.2873
✅ Best model saved!



📘 Epoch 3/30
Train Loss: 1.4335, Train Acc: 0.6465
Val   Loss: 1.5278, Val   Acc: 0.6011
✅ Best model saved!



📘 Epoch 4/30
Train Loss: 0.8524, Train Acc: 0.7809
Val   Loss: 1.1751, Val   Acc: 0.6756
✅ Best model saved!



📘 Epoch 5/30
Train Loss: 0.5701, Train Acc: 0.8501
Val   Loss: 1.2341, Val   Acc: 0.6657


...


📘 Epoch 26/30
Train Loss: 0.0390, Train Acc: 0.9893
Val   Loss: 0.6206, Val   Acc: 0.8621
✅ Best model saved!



📘 Epoch 27/30
Train Loss: 0.0504, Train Acc: 0.9858
Val   Loss: 0.7745, Val   Acc: 0.8301



📘 Epoch 28/30
Train Loss: 0.0475, Train Acc: 0.9868
Val   Loss: 0.7067, Val   Acc: 0.8500



📘 Epoch 29/30
Train Loss: 0.0539, Train Acc: 0.9845
Val   Loss: 0.7265, Val   Acc: 0.8437



📘 Epoch 30/30
Train Loss: 0.0334, Train Acc: 0.9919
Val   Loss: 0.8747, Val   Acc: 0.8174


In [8]:
# # ✅ 1. 라이브러리 임포트
# import os
# import pandas as pd
# import torch
# import torch.nn as nn
# from torch.utils.data import Dataset, DataLoader
# from torchvision import models, transforms
# from PIL import Image

# # ✅ 2. 디바이스 설정
# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# # ✅ 3. 전처리 정의 (학습과 동일하게!)
# transform_test = transforms.Compose([
#     transforms.Resize((224, 224)),
#     transforms.ToTensor(),
#     transforms.Normalize(mean=[0.485, 0.456, 0.406],
#                          std=[0.229, 0.224, 0.225])
# ])

# # ✅ 4. Test Dataset 정의
# class TestDataset(Dataset):
#     def __init__(self, dataframe, root_dir, transform=None):
#         self.dataframe = dataframe
#         self.root_dir = root_dir
#         self.transform = transform

#     def __len__(self):
#         return len(self.dataframe)

#     def __getitem__(self, idx):
#         img_name = os.path.join(self.root_dir, self.dataframe.iloc[idx]['img_path'])
#         image = Image.open(img_name).convert("RGB")
#         if self.transform:
#             image = self.transform(image)
#         return image

# # ✅ 5. 경로 & 데이터 불러오기
# test_root = "/kaggle/input/hectocar/test"
# test_df = pd.read_csv("/kaggle/input/hectocar/test.csv")
# # ✅ test_df 불러온 직후 이 코드 한 줄 추가
# test_df['img_path'] = test_df['img_path'].str.replace("test/", "", regex=False)
# sample_sub = pd.read_csv("/kaggle/input/hectocar/sample_submission.csv")

# # ✅ 6. 데이터로더 정의
# test_dataset = TestDataset(test_df, test_root, transform=transform_test)
# test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

# # ✅ 7. 모델 로드
# model = models.resnet18(weights=None)
# model.fc = nn.Linear(model.fc.in_features, 396)
# model.load_state_dict(torch.load("best_model_396.pth", map_location=device))
# model = model.to(device)
# model.eval()

# # ✅ 8. 추론 실행
# all_preds = []

# with torch.no_grad():
#     for images in test_loader:
#         images = images.to(device)
#         outputs = model(images)
#         probs = torch.softmax(outputs, dim=1)
#         all_preds.extend(probs.cpu().numpy())

# # ✅ 9. 결과를 sample_submission 형식에 맞게 저장
# submission = pd.DataFrame(all_preds, columns=sample_sub.columns[1:])
# submission.insert(0, 'ID', test_df['ID'])
# submission.to_csv("submission.csv", index=False)

# print("🎉 제출 파일 'submission.csv' 생성 완료!")


🎉 제출 파일 'submission.csv' 생성 완료!


In [10]:
import os
import pandas as pd
import numpy as np
from PIL import Image
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.models as models
import torchvision.transforms as transforms

# ✅ 환경 설정
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
test_root = "/kaggle/input/hectocar/test"
model_path = "./best_model_396.pth"

# ✅ 전처리 정의 (학습 시와 동일해야 함)
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

# ✅ 데이터셋 정의
class TestDataset(Dataset):
    def __init__(self, dataframe, root_dir, transform=None):
        self.dataframe = dataframe
        self.root_dir = root_dir
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        img_path = os.path.join(self.root_dir, self.dataframe.iloc[idx]['img_path'])
        image = Image.open(img_path).convert("RGB")
        if self.transform:
            image = self.transform(image)
        return image

# ✅ 데이터 불러오기
test_df = pd.read_csv("/kaggle/input/hectocar/test.csv")
test_df['img_path'] = test_df['img_path'].str.replace("test/", "", regex=False)
sample_sub = pd.read_csv("/kaggle/input/hectocar/sample_submission.csv")

# ✅ class 순서 맞추기
class_order = sample_sub.columns[1:]  # 'ID' 제외한 클래스 열들
class_to_index = {cls_name: i for i, cls_name in enumerate(class_order)}

# ✅ 모델 불러오기
model = models.resnet18(weights=None)
model.fc = nn.Linear(model.fc.in_features, len(class_order))  # 클래스 수 = 396
model.load_state_dict(torch.load(model_path, map_location=device))
model.to(device)
model.eval()

# ✅ 테스트셋 준비
test_dataset = TestDataset(test_df, test_root, transform=transform)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

# ✅ 추론 및 확률 저장
all_probs = []

with torch.no_grad():
    for images in tqdm(test_loader, desc="🔍 Inference"):
        images = images.to(device)
        outputs = model(images)
        probs = F.softmax(outputs, dim=1).cpu().numpy()
        all_probs.extend(probs)

# ✅ DataFrame으로 변환 (클래스 순서에 맞게)
submission = pd.DataFrame(all_probs, columns=class_order)
submission.insert(0, "ID", test_df["ID"])  # ID 컬럼 맨 앞에 삽입

# ✅ 저장
submission.to_csv("submission.csv", index=False)
print("✅ 'submission.csv' 저장 완료 🎯")


🔍 Inference: 100%|██████████| 130/130 [01:14<00:00,  1.75it/s]


✅ 'submission.csv' 저장 완료 🎯
